# Regression Adjustments

## Load Data

In [311]:
import os
import pandas as pd
import numpy as np

os.chdir('/Users/janlinzner/Projects/Master-Thesis-Spatial-Proximity-Venture-Capital')

In [312]:
data = pd.read_csv('data/sets-for-r/data.csv')

## Outlier cutting

In [313]:
chosen_variables = ['number_of_founders', 'avg_age_of_seed_investors', 'total_seed_funding_m']

lower_quantile = data[chosen_variables].quantile(0.01)
upper_quantile = data[chosen_variables].quantile(0.99)

for variable in chosen_variables:
    lower_bound = lower_quantile[variable]
    upper_bound = upper_quantile[variable]
    data = data[(data[variable] >= lower_bound) & (data[variable] <= upper_bound)]

In [314]:
data

,company_id,organization_name,organization_name_url,headquarters_country,latitude,longitude,founded_year,number_of_founders,b2b_binary,hub_binary,...,number_seed_investors_same_city_30,number_seed_lead_investors_same_city_30,number_seed_investors_same_city_40,number_seed_lead_investors_same_city_40,number_seed_investors_same_city_50,number_seed_lead_investors_same_city_50,number_seed_investors_same_city_70,number_seed_lead_investors_same_city_70,number_seed_investors_same_city_100,number_seed_lead_investors_same_city_100
0,16452,Traffic Observation via Management,https://www.crunchbase.com/organization/traffi...,United Kingdom,53.407199,-2.991680,2007,1.0,True,False,...,1,0,1,0,1,0,1,0,1,0
1,9341,Secusmart,https://www.crunchbase.com/organization/secusmart,Germany,51.225402,6.776314,2007,1.0,True,False,...,0,0,0,0,0,0,0,0,0,0
2,1893,Adipsys,https://www.crunchbase.com/organization/adipsys,France,43.641141,7.008625,2007,1.0,True,False,...,0,0,0,0,0,0,0,0,0,0
3,16174,Crescendo Biologics,https://www.crunchbase.com/organization/cresce...,United Kingdom,52.205531,0.118664,2007,1.0,True,True,...,2,0,2,0,2,0,2,0,3,0
4,1894,ActiveEon,https://www.crunchbase.com/organization/activeeon,France,43.619523,7.051816,2007,1.0,True,False,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10265,18186,FLOWBIO,https://www.crunchbase.com/organization/flow-bio,United Kingdom,51.489334,-0.144055,2020,2.0,False,True,...,2,0,2,0,2,0,2,0,2,0
10266,18183,Yoof Agency,https://www.crunchbase.com/organization/yoof-a...,United Kingdom,51.489334,-0.144055,2020,1.0,False,True,...,0,0,0,0,0,0,0,0,0,0
10267,18182,Noah,https://www.crunchbase.com/organization/noah-7ab8,United Kingdom,51.489334,-0.144055,2020,1.0,True,True,...,0,0,0,0,0,0,0,0,0,0
10268,19011,Hackrate,https://www.crunchbase.com/organization/hackrate,Hungary,47.497879,19.040238,2020,2.0,True,False,...,4,3,4,3,4,3,4,3,4,3


## Kilometer variables in 1000km

In [315]:
variables_to_divide = [
    'distance_to_hub',
    'avg_seed_investor_distance',
    'minimum_seed_vc_distance',
    'max_seed_investor_distance',
    'minimum_seed_vc_distance_non_lead',
    'avg_seed_vc_distance_non_lead',
    'maximum_seed_vc_distance_non_lead',
    'minimum_lead_seed_vc_distance',
    'avg_lead_seed_vc_distance',
    'maximum_lead_seed_vc_distance',
    'avg_seed_investor_pairwise_distance',
    'min_seed_investor_pairwise_distance',
    'max_seed_investor_pairwise_distance'
]

for variable in variables_to_divide:
    if variable in data.columns:
        data[variable] = data[variable] / 1000

## Save overall set

In [316]:
data.to_csv('data/sets-for-r/regression_sets/df_europe.csv', index=False)

## Country sets 

In [317]:
country_dataframes = {country: data[data['headquarters_country'] == country] for country in data['headquarters_country'].unique()}

df_ger = country_dataframes['Germany']
df_uk = country_dataframes['United Kingdom']
df_fr = country_dataframes['France']
df_ita = country_dataframes['Italy']
df_sp = country_dataframes['Spain']
df_nl = country_dataframes['The Netherlands']
df_be = country_dataframes['Belgium']
df_nordics = pd.concat([country_dataframes['Denmark'], 
                        country_dataframes['Finland'], 
                        country_dataframes['Norway']])
df_east = pd.concat([country_dataframes['Czech Republic'], 
                     country_dataframes['Latvia'], 
                     country_dataframes['Poland'], 
                     country_dataframes['Slovakia (Slovak Republic)'], 
                     country_dataframes['Estonia'], 
                     country_dataframes['Lithuania'], 
                     country_dataframes['Hungary']])

df_dach = pd.concat([country_dataframes['Germany'], 
                     country_dataframes['Austria'], 
                     country_dataframes['Switzerland']])

df_big3 = pd.concat([country_dataframes['Germany'], 
                     country_dataframes['United Kingdom'], 
                     country_dataframes['France']])

In [318]:
dataframes_to_save = {
    'df_ger': df_ger,
    'df_uk': df_uk,
    'df_fr': df_fr,
    'df_ita': df_ita,
    'df_sp': df_sp,
    'df_nl': df_nl,
    'df_be': df_be,
    'df_nordics': df_nordics,
    'df_east': df_east,
    'df_dach': df_dach,
    'df_big3': df_big3
}

for name, dataframe in dataframes_to_save.items():
    dataframe.to_csv(f'data/sets-for-r/regression_sets/{name}.csv', index=False)

## Sector Datasets

In [319]:
sector_columns = [
    'energy', 'materials', 'industrials', 'consumer_discretionary', 'consumer_staples', 
    'health_care', 'financials', 'information_technology', 'communication_services', 
    'utilities', 'real_estate', 'other'
]

sector_datasets = {sector: data[data[sector] == True] for sector in sector_columns}

In [320]:
df_energy = sector_datasets['energy']
df_materials = sector_datasets['materials']
df_industrials = sector_datasets['industrials']
df_consumer_discretionary = sector_datasets['consumer_discretionary']
df_consumer_staples = sector_datasets['consumer_staples']
df_health_care = sector_datasets['health_care']
df_financials = sector_datasets['financials']
df_information_technology = sector_datasets['information_technology']
df_communication_services = sector_datasets['communication_services']
df_utilities = sector_datasets['utilities']
df_real_estate = sector_datasets['real_estate']
df_other = sector_datasets['other']

In [321]:
for sector, dataframe in sector_datasets.items():
    dataframe.to_csv(f'data/sets-for-r/regression_sets/df_{sector}.csv', index=False)